In [0]:
S1_PATH = "abfss://silverlayer@cryptodl.dfs.core.windows.net/Batch_Data_Source_1/"
S2_PATH = "abfss://silverlayer@cryptodl.dfs.core.windows.net/JSON_Streaming_Data_Source_2/"
S3_PATH = "abfss://silverlayer@cryptodl.dfs.core.windows.net/CSV_Streaming_Data_Source_3/"

 
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
 
spark = SparkSession.builder.getOrCreate()
 
 
s1 = spark.read.format("delta").load(S1_PATH)
s2 = spark.read.format("delta").load(S2_PATH)
s3 = spark.read.format("delta").load(S3_PATH)
 

s1 = s1.withColumnRenamed("Symbol", "symbol") \
       .withColumn("trade_date", F.to_date(F.col("trade_date")))
 
s2 = s2.withColumn("trade_date", F.to_date(F.col("trade_date")))
s3 = s3.withColumn("trade_date", F.to_date(F.col("trade_date")))
 

s1.createOrReplaceTempView("silver_prices")
s2.createOrReplaceTempView("silver_social")
s3.createOrReplaceTempView("silver_metrics")
 
print("All three sources loaded")
print(f"  S1 rows: {s1.count():,}")
print(f"  S2 rows: {s2.count():,}")
print(f"  S3 rows: {s3.count():,}")


In [0]:

print("\n=== TEST 1: Symbol Overlap ===")
spark.sql("""
    SELECT
        p.symbol,
        COUNT(DISTINCT p.trade_date)  AS price_dates,
        COUNT(DISTINCT s.trade_date)  AS social_dates,
        COUNT(DISTINCT m.trade_date)  AS metric_dates
    FROM silver_prices  p
    LEFT JOIN silver_social   s ON p.symbol = s.symbol
    LEFT JOIN silver_metrics  m ON p.symbol = m.symbol
    GROUP BY p.symbol
    ORDER BY p.symbol
""").show(30)

In [0]:
print("\n=== TEST 2: Date Overlap (S1 ∩ S2 ∩ S3) ===")
spark.sql("""
    SELECT
        p.symbol,
        p.trade_date,
        p.close_price AS price_close,
        s.sentiment_score,
        s.sentiment_label,
        m.active_addresses,
        m.net_exchange_flow,
        m.flow_signal
    FROM silver_prices  p
    INNER JOIN silver_social   s ON p.symbol = s.symbol AND p.trade_date = s.trade_date
    INNER JOIN silver_metrics  m ON p.symbol = m.symbol AND p.trade_date = m.trade_date
    ORDER BY p.symbol, p.trade_date
    LIMIT 20
""").show(20, truncate=False)


In [0]:

print("\n=== TEST 3: INNER JOIN Row Counts ===")
s1_s2 = spark.sql("""
    SELECT COUNT(*) as s1_s2_matches
    FROM silver_prices p
    INNER JOIN silver_social s
    ON p.symbol = s.symbol AND p.trade_date = s.trade_date
""")
 
s1_s3 = spark.sql("""
    SELECT COUNT(*) as s1_s3_matches
    FROM silver_prices p
    INNER JOIN silver_metrics m
    ON p.symbol = m.symbol AND p.trade_date = m.trade_date
""")
 
all_three = spark.sql("""
    SELECT COUNT(*) as all_three_matches
    FROM silver_prices p
    INNER JOIN silver_social   s ON p.symbol = s.symbol AND p.trade_date = s.trade_date
    INNER JOIN silver_metrics  m ON p.symbol = m.symbol AND p.trade_date = m.trade_date
""")
 
s1_s2.show()
s1_s3.show()
all_three.show()

In [0]:
print("\n=== TEST 4: Null Check on Join Keys ===")
spark.sql("""
    SELECT 'S1' as source, COUNT(*) as null_symbol FROM silver_prices  WHERE symbol IS NULL OR trade_date IS NULL
    UNION ALL
    SELECT 'S2',           COUNT(*)                FROM silver_social   WHERE symbol IS NULL OR trade_date IS NULL
    UNION ALL
    SELECT 'S3',           COUNT(*)                FROM silver_metrics  WHERE symbol IS NULL OR trade_date IS NULL
""").show()

In [0]:
print("\n=== TEST 2: Date Overlap (S1 ∩ S2 ∩ S3) ===")
spark.sql("""
    SELECT
        p.symbol,
        p.trade_date,
        p.close_price,
        s.sentiment_score,
        s.sentiment_label,
        m.active_addresses,
        m.net_exchange_flow,
        m.flow_signal
    FROM silver_prices  p
    INNER JOIN silver_social   s ON p.symbol = s.symbol AND p.trade_date = s.trade_date
    INNER JOIN silver_metrics  m ON p.symbol = m.symbol AND p.trade_date = m.trade_date
    ORDER BY p.symbol, p.trade_date
    LIMIT 20
""").show(20, truncate=False)



In [0]:
print("\n=== TEST 5: Gold Layer Preview (what the model will see) ===")
gold_preview = spark.sql("""
    SELECT
        p.symbol,
        p.trade_date,
        p.close_price,
        p.high_price,
        p.low_price,
        p.open_price,
        p.Volume          AS volume_usd,
        p.market_cap      AS market_cap_usd,
        s.sentiment_score,
        s.sentiment_label,
        s.engagement_score,
        m.active_addresses,
        m.transaction_count,
        m.network_fees,
        m.net_exchange_flow,
        m.flow_signal
    FROM silver_prices  p
    INNER JOIN silver_social   s ON p.symbol = s.symbol AND p.trade_date = s.trade_date
    INNER JOIN silver_metrics  m ON p.symbol = m.symbol AND p.trade_date = m.trade_date
    ORDER BY p.symbol, p.trade_date
""")

print(f"Gold preview rows: {gold_preview.count():,}")
gold_preview.printSchema()
display(gold_preview.limit(20))


In [0]:
s1 = spark.read.format("delta").load(
    "abfss://silverlayer@cryptodl.dfs.core.windows.net/Batch_Data_Source_1/"
)

combos = (
    s1.select("symbol", "trade_date")
      .distinct()
      .limit(50)
      .collect()
)

print("Combos to use:")
for c in combos[:5]:
    print(f"  {c['symbol']} | {c['trade_date']}")


import json
combos_list = [{"symbol": c["symbol"], "trade_date": str(c["trade_date"])} for c in combos]
print(f"Total: {len(combos_list)} combos")

In [0]:
s2 = spark.read.format("delta").load("abfss://silverlayer@cryptodl.dfs.core.windows.net/JSON_Streaming_Data_Source_2/")
s3 = spark.read.format("delta").load("abfss://silverlayer@cryptodl.dfs.core.windows.net/CSV_Streaming_Data_Source_3/")

# Show S2 sample dates
print("S2 sample symbol+date:")
s2.select("symbol","trade_date").distinct().show(5)

# Show S3 sample dates  
print("S3 sample symbol+date:")
s3.select("symbol","trade_date").distinct().show(5)

# Check overlap between S2 and S3
overlap = s2.select("symbol","trade_date").intersect(s3.select("symbol","trade_date"))
print(f"S2 ∩ S3 = {overlap.count()}")